In [116]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go
import plotly.express as px
from tqdm import tqdm


In [ ]:
days = [-1]  
df_all = []

for day in days:
    try:
        df_day = pd.read_csv(f"prices_round_2_day_{day}.csv", sep=";", header=0)
        # Add a day identifier
        df_day['day'] = day
        df_all.append(df_day)
    except FileNotFoundError:
        print(f"Warning: File for day {day} not found. Skipping.")
    except Exception as e:
        print(f"Error processing day {day}: {e}")

# Check if any dataframes were loaded
if not df_all:
    raise ValueError("No dataframes were loaded. Check your file paths.")

# Concatenate all days
df = pd.concat(df_all, ignore_index=True)

# Create a continuous time index
# First, sort by day and timestamp
df = df.sort_values(by=['day', 'timestamp'])

# Create a continuous timestamp
df['continuous_time'] = range(len(df))

# Optional: Print some basic info about the loaded data
print(f"Total rows loaded: {len(df)}")
print(f"Unique products: {df['product'].unique()}")

In [ ]:
import plotly.graph_objs as go
import numpy as np

# Filter and prepare data
df_picnic_basket_1 = df[df['product'] == "PICNIC_BASKET1"].copy()

# Calculate VWAP with error handling
def calculate_vwap(row):
    ask_volume = row['ask_volume_1']
    bid_volume = row['bid_volume_1']
    ask_price = row['ask_price_1']
    bid_price = row['bid_price_1']
    
    # Avoid division by zero
    if ask_volume + bid_volume > 0:
        return (bid_price * ask_volume + ask_price * bid_volume) / (ask_volume + bid_volume)
    else:
        return np.nan

df_picnic_basket_1['VWAP'] = df_picnic_basket_1.apply(calculate_vwap, axis=1)

# Remove NaN values
df_picnic_basket_1_clean = df_picnic_basket_1.dropna(subset=['VWAP'])

# Create figure
fig = go.Figure()

# Add VWAP trace
fig.add_trace(go.Scatter(
    x=df_picnic_basket_1_clean['continuous_time'], 
    y=df_picnic_basket_1_clean['VWAP'], 
    mode='lines', 
    name='VWAP PICNIC Basket Price', 
    line=dict(color='blue')
))

# Update layout
fig.update_layout(
    title_text="VWAP PICNIC BASKET 1",
    xaxis_title="Continuous Time",
    yaxis_title="Price",
    template='plotly_white'  # Optional: gives a clean look
)

# Add grid and improve readability
fig.update_xaxes(showgrid=True, gridwidth=1, gridcolor='lightgray')
fig.update_yaxes(showgrid=True, gridwidth=1, gridcolor='lightgray')

# Show the plot
fig.show()

# Optional: Print some statistics
print("VWAP Statistics for PICNIC BASKET1:")
print(f"Mean VWAP: {df_picnic_basket_1_clean['VWAP'].mean():.2f}")
print(f"Median VWAP: {df_picnic_basket_1_clean['VWAP'].median():.2f}")
print(f"Min VWAP: {df_picnic_basket_1_clean['VWAP'].min():.2f}")
print(f"Max VWAP: {df_picnic_basket_1_clean['VWAP'].max():.2f}")

In [ ]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

def calculate_swmid(df):
    """Calculate Smart Weighted Mid Price with error handling"""
    try:
        swmid = (df['bid_price_1'] * df['ask_volume_1'] + df['ask_price_1'] * df['bid_volume_1']) / (df['ask_volume_1'] + df['bid_volume_1'])
        return swmid
    except ZeroDivisionError:
        print("Warning: Zero division encountered in SWMID calculation")
        return df['mid_price']  # Fallback to mid_price

# Filter and prepare data
df_croissant = df[df['product'] == 'CROISSANTS'].copy()
df_jam = df[df['product'] == 'JAMS'].copy()
df_djembe = df[df['product'] == 'DJEMBES'].copy()
df_basket_1 = df[df['product'] == 'PICNIC_BASKET1'].copy()
df_basket_2 = df[df['product'] == 'PICNIC_BASKET2'].copy()

# Calculate smart weighted mid price for each product
df_croissant['swmid'] = calculate_swmid(df_croissant)
df_jam['swmid'] = calculate_swmid(df_jam)
df_djembe['swmid'] = calculate_swmid(df_djembe)
df_basket_1['swmid'] = calculate_swmid(df_basket_1)
df_basket_2['swmid'] = calculate_swmid(df_basket_2)

# Create synthetic prices for PICNIC_BASKET1 (6 CROISSANTS, 3 JAMS, 1 DJEMBE)
df_synthetic_1 = pd.DataFrame({
    'timestamp': df_croissant['timestamp'].to_numpy(),
    'continuous_time': df_croissant['continuous_time'].to_numpy(),
    'day': df_croissant['day'].to_numpy(),
    'bid_price_1': df_croissant['bid_price_1'].to_numpy() * 6 + df_jam['bid_price_1'].to_numpy() * 3 + df_djembe['bid_price_1'].to_numpy(),
    'bid_volume_1': np.min(np.array([
        df_croissant['bid_volume_1'].to_numpy(), 
        df_jam['bid_volume_1'].to_numpy(), 
        df_djembe['bid_volume_1'].to_numpy()
    ]), axis=0),
    'ask_price_1': df_croissant['ask_price_1'].to_numpy() * 6 + df_jam['ask_price_1'].to_numpy() * 3 + df_djembe['ask_price_1'].to_numpy(),
    'ask_volume_1': np.min(np.array([
        df_croissant['ask_volume_1'].to_numpy(), 
        df_jam['ask_volume_1'].to_numpy(), 
        df_djembe['ask_volume_1'].to_numpy()
    ]), axis=0),
    'mid_price': df_croissant['mid_price'].to_numpy() * 6 + df_jam['mid_price'].to_numpy() * 3 + df_djembe['mid_price'].to_numpy(),
    'swmid': df_croissant['swmid'].to_numpy() * 6 + df_jam['swmid'].to_numpy() * 3 + df_djembe['swmid'].to_numpy()
})

# Create synthetic prices for PICNIC_BASKET2 (4 CROISSANTS, 2 JAMS)
df_synthetic_2 = pd.DataFrame({
    'timestamp': df_croissant['timestamp'].to_numpy(),
    'continuous_time': df_croissant['continuous_time'].to_numpy(),
    'day': df_croissant['day'].to_numpy(),
    'bid_price_1': df_croissant['bid_price_1'].to_numpy() * 4 + df_jam['bid_price_1'].to_numpy() * 2,
    'bid_volume_1': np.min(np.array([
        df_croissant['bid_volume_1'].to_numpy() // 4, 
        df_jam['bid_volume_1'].to_numpy() // 2
    ]), axis=0),
    'ask_price_1': df_croissant['ask_price_1'].to_numpy() * 4 + df_jam['ask_price_1'].to_numpy() * 2,
    'ask_volume_1': np.min(np.array([
        df_croissant['ask_volume_1'].to_numpy() // 4, 
        df_jam['ask_volume_1'].to_numpy() // 2
    ]), axis=0),
    'mid_price': df_croissant['mid_price'].to_numpy() * 4 + df_jam['mid_price'].to_numpy() * 2,
    'swmid': df_croissant['swmid'].to_numpy() * 4 + df_jam['swmid'].to_numpy() * 2
})

# Plot with continuous time on x-axis
fig = make_subplots(specs=[[{"secondary_y": True}]])

# Calculate spread between Picnic Basket and Synthetic
spread = df_basket_1['swmid'].to_numpy() - df_synthetic_1['swmid'].to_numpy()

# Use continuous_time for x-axis
fig.add_trace(go.Scatter(
    x=df_basket_1['continuous_time'], 
    y=df_basket_1['swmid'], 
    mode='lines', 
    name='Picnic Basket 1 SWMID', 
    line=dict(color='blue')
), secondary_y=False)

fig.add_trace(go.Scatter(
    x=df_synthetic_1['continuous_time'], 
    y=df_synthetic_1['swmid'], 
    mode='lines', 
    name='Synthetic SWMID', 
    line=dict(color='red')
), secondary_y=True)

# Add day separators
for day in days[1:]:  # Skip the first day
    # Find first timestamp of this day
    day_start = df[df['day'] == day]['continuous_time'].min()
    fig.add_vline(x=day_start, line_dash="dash", line_color="gray", 
                  annotation_text=f"Day {day}", annotation_position="top right")

fig.update_layout(
    title_text="Synthetic and Picnic Basket 1 SWMID - All Days",
    template='plotly_white'
)
fig.update_xaxes(title_text="Continuous Time")
fig.update_yaxes(title_text="Picnic Basket SWMID", secondary_y=False)
fig.update_yaxes(title_text="Synthetic SWMID", secondary_y=True)

fig.show()

# Optional: Print some statistics about the spread
print("\nSpread Analysis:")
print(f"Mean Spread: {np.mean(spread):.2f}")
print(f"Median Spread: {np.median(spread):.2f}")
print(f"Spread Standard Deviation: {np.std(spread):.2f}")
print(f"Min Spread: {np.min(spread):.2f}")
print(f"Max Spread: {np.max(spread):.2f}")

In [ ]:
# Create a spread dataframe
spread_df = pd.DataFrame({
    'continuous_index': df_basket_1['continuous_time'],  # Use continuous_time instead of continuous_index
    'spread': df_basket_1['swmid'] - df_synthetic_1['swmid']
})

# Create figure with improved styling
fig = go.Figure()

# Add spread trace
fig.add_trace(go.Scatter(
    x=spread_df['continuous_index'],
    y=spread_df['spread'],
    mode='lines',
    name='Basket-Synthetic Spread',
    line=dict(color='purple', width=2)  # Slightly thicker line
))

# Calculate statistical metrics
mean_spread = spread_df['spread'].mean()
std_spread = spread_df['spread'].std()

# Add statistical annotations
fig.update_layout(
    title_text="Picnic Basket 1 - Synthetic Spread",
    template='plotly_white',  # Clean, modern template
    annotations=[
        dict(
            x=0.02,
            y=0.95,
            xref='paper',
            yref='paper',
            text=f"Mean: {mean_spread:.2f}<br>Std Dev: {std_spread:.2f}",
            showarrow=False,
            bgcolor='rgba(255,255,255,0.7)',
            bordercolor='lightgray',
            borderwidth=1,
            borderpad=5
        )
    ]
)

# Update axes
fig.update_xaxes(
    title_text="Trading Period", 
    showgrid=True, 
    gridwidth=1, 
    gridcolor='lightgray'
)
fig.update_yaxes(
    title_text="Spread", 
    showgrid=True, 
    gridwidth=1, 
    gridcolor='lightgray'
)

# Add horizontal lines for mean and standard deviation bands
fig.add_hline(
    y=mean_spread, 
    line_dash="dash", 
    line_color="green", 
    annotation_text=f"Mean: {mean_spread:.2f}"
)
fig.add_hline(
    y=mean_spread + std_spread, 
    line_dash="dot", 
    line_color="red", 
    annotation_text=f"+1σ: {mean_spread + std_spread:.2f}"
)
fig.add_hline(
    y=mean_spread - std_spread, 
    line_dash="dot", 
    line_color="red", 
    annotation_text=f"-1σ: {mean_spread - std_spread:.2f}"
)

# Show the plot
fig.show()

# Optional: Print detailed spread statistics
print("\nSpread Analysis:")
print(f"Mean Spread: {mean_spread:.2f}")
print(f"Median Spread: {spread_df['spread'].median():.2f}")
print(f"Spread Standard Deviation: {std_spread:.2f}")
print(f"Min Spread: {spread_df['spread'].min():.2f}")
print(f"Max Spread: {spread_df['spread'].max():.2f}")
print(f"Spread Range: {spread_df['spread'].max() - spread_df['spread'].min():.2f}")

In [ ]:
from plotly.subplots import make_subplots
import plotly.graph_objects as go
import numpy as np

# Create subplot figure
fig = make_subplots(specs=[[{"secondary_y": True}]])

# Use continuous_time instead of timestamp for better visualization
fig.add_trace(
    go.Scatter(
        x=df_basket_2['continuous_time'], 
        y=df_basket_2['swmid'], 
        mode='lines', 
        name='Gift Basket SWMID', 
        line=dict(color='blue', width=2)
    ), 
    secondary_y=False
)

fig.add_trace(
    go.Scatter(
        x=df_synthetic_2['continuous_time'], 
        y=df_synthetic_2['swmid'], 
        mode='lines', 
        name='Synthetic SWMID', 
        line=dict(color='red', width=2)
    ), 
    secondary_y=True
)

# Calculate and add spread trace
spread = df_basket_2['swmid'].to_numpy() - df_synthetic_2['swmid'].to_numpy()
spread_mean = np.mean(spread)
spread_std = np.std(spread)

# Update layout with more informative details
fig.update_layout(
    title_text="Synthetic and Gift Basket SWMID 2",
    template='plotly_white',
    annotations=[
        dict(
            x=0.02,
            y=0.95,
            xref='paper',
            yref='paper',
            text=f"Spread Mean: {spread_mean:.2f}<br>Spread Std Dev: {spread_std:.2f}",
            showarrow=False,
            bgcolor='rgba(255,255,255,0.7)',
            bordercolor='lightgray',
            borderwidth=1,
            borderpad=5
        )
    ]
)

# Improve axis labels
fig.update_xaxes(
    title_text="Continuous Time", 
    showgrid=True, 
    gridwidth=1, 
    gridcolor='lightgray'
)
fig.update_yaxes(
    title_text="Gift Basket SWMID", 
    secondary_y=False,
    showgrid=True, 
    gridwidth=1, 
    gridcolor='lightgray'
)
fig.update_yaxes(
    title_text="Synthetic SWMID", 
    secondary_y=True,
    showgrid=True, 
    gridwidth=1, 
    gridcolor='lightgray'
)

# Show the plot
fig.show()

# Print spread analysis
print("\nSpread Analysis for Gift Basket 2:")
print(f"Mean Spread: {spread_mean:.2f}")
print(f"Spread Standard Deviation: {spread_std:.2f}")
print(f"Min Spread: {np.min(spread):.2f}")
print(f"Max Spread: {np.max(spread):.2f}")
print(f"Spread Range: {np.max(spread) - np.min(spread):.2f}")

In [ ]:
# Recreate spread as a DataFrame if it's not already one
spread = pd.DataFrame({
    'continuous_index': df_basket_1['continuous_time'],
    'spread': df_basket_1['swmid'] - df_synthetic_1['swmid']
})

# Calculate rolling standard deviation
spread['std30'] = spread['spread'].rolling(window=30, min_periods=1).std()

# Alternative calculation methods
spread['std_explicit'] = spread['spread'].rolling(window=30, min_periods=1).std()
spread['rmsd_376'] = spread['spread'].rolling(
    window=30, 
    min_periods=1
).apply(
    lambda x: np.sqrt(np.mean((x - 376)**2)) if len(x) > 0 else np.nan
)

# Print some information to verify
print(spread.info())
print(spread.head())

In [85]:
# 2. Process all the data
df_croissant = df[df['product'] == 'CROISSANTS'].copy()
df_jam = df[df['product'] == 'JAMS'].copy()
df_djembe = df[df['product'] == 'DJEMBES'].copy()
df_basket_1 = df[df['product'] == 'PICNIC_BASKET1'].copy()
df_basket_2 = df[df['product'] == 'PICNIC_BASKET2'].copy()

# Calculate smart weighted mid price for each product
df_croissant['swmid'] = (df_croissant['bid_price_1'] * df_croissant['ask_volume_1'] + df_croissant['ask_price_1'] * df_croissant['bid_volume_1']) / (df_croissant['ask_volume_1'] + df_croissant['bid_volume_1'])
df_jam['swmid'] = (df_jam['bid_price_1'] * df_jam['ask_volume_1'] + df_jam['ask_price_1'] * df_jam['bid_volume_1']) / (df_jam['ask_volume_1'] + df_jam['bid_volume_1'])
df_djembe['swmid'] = (df_djembe['bid_price_1'] * df_djembe['ask_volume_1'] + df_djembe['ask_price_1'] * df_djembe['bid_volume_1']) / (df_djembe['ask_volume_1'] + df_djembe['bid_volume_1'])
df_basket_1['swmid'] = (df_basket_1['bid_price_1'] * df_basket_1['ask_volume_1'] + df_basket_1['ask_price_1'] * df_basket_1['bid_volume_1']) / (df_basket_1['ask_volume_1'] + df_basket_1['bid_volume_1'])
df_basket_2['swmid'] = (df_basket_2['bid_price_1'] * df_basket_2['ask_volume_1'] + df_basket_2['ask_price_1'] * df_basket_2['bid_volume_1']) / (df_basket_2['ask_volume_1'] + df_basket_2['bid_volume_1'])

# Create synthetic prices for PICNIC_BASKET1 (6 CROISSANTS, 3 JAMS, 1 DJEMBE)
df_synthetic_1 = pd.DataFrame({
    'timestamp': df_croissant['timestamp'].to_numpy(),
    'day': df_croissant['day'].to_numpy(),
    'bid_price_1': df_croissant['bid_price_1'].to_numpy() * 6 + df_jam['bid_price_1'].to_numpy() * 3 + df_djembe['bid_price_1'].to_numpy(),
    'bid_volume_1': np.min(np.array([
        df_croissant['bid_volume_1'].to_numpy(), 
        df_jam['bid_volume_1'].to_numpy(), 
        df_djembe['bid_volume_1'].to_numpy()
    ]), axis=0),
    'ask_price_1': df_croissant['ask_price_1'].to_numpy() * 6 + df_jam['ask_price_1'].to_numpy() * 3 + df_djembe['ask_price_1'].to_numpy(),
    'ask_volume_1': np.min(np.array([
        df_croissant['ask_volume_1'].to_numpy(), 
        df_jam['ask_volume_1'].to_numpy(), 
        df_djembe['ask_volume_1'].to_numpy()
    ]), axis=0),
    'mid_price': df_croissant['mid_price'].to_numpy() * 6 + df_jam['mid_price'].to_numpy() * 3 + df_djembe['mid_price'].to_numpy(),
    'swmid': df_croissant['swmid'].to_numpy() * 6 + df_jam['swmid'].to_numpy() * 3 + df_djembe['swmid'].to_numpy()
})

# Create synthetic prices for PICNIC_BASKET2 (4 CROISSANTS, 2 JAMS)
df_synthetic_2 = pd.DataFrame({
    'timestamp': df_croissant['timestamp'].to_numpy(),
    'day': df_croissant['day'].to_numpy(),
    'bid_price_1': df_croissant['bid_price_1'].to_numpy() * 4 + df_jam['bid_price_1'].to_numpy() * 2,
    'bid_volume_1': np.min(np.array([
        df_croissant['bid_volume_1'].to_numpy(), 
        df_jam['bid_volume_1'].to_numpy()
    ]), axis=0),
    'ask_price_1': df_croissant['ask_price_1'].to_numpy() * 4 + df_jam['ask_price_1'].to_numpy() * 2,
    'ask_volume_1': np.min(np.array([
        df_croissant['ask_volume_1'].to_numpy(), 
        df_jam['ask_volume_1'].to_numpy()
    ]), axis=0),
    'mid_price': df_croissant['mid_price'].to_numpy() * 4 + df_jam['mid_price'].to_numpy() * 2,
    'swmid': df_croissant['swmid'].to_numpy() * 4 + df_jam['swmid'].to_numpy() * 2
})

# Sort by day and timestamp to ensure chronological order
df_basket_1 = df_basket_1.sort_values(['day', 'timestamp']).reset_index(drop=True)
df_synthetic_1 = df_synthetic_1.sort_values(['day', 'timestamp']).reset_index(drop=True)
df_basket_2 = df_basket_2.sort_values(['day', 'timestamp']).reset_index(drop=True)
df_synthetic_2 = df_synthetic_2.sort_values(['day', 'timestamp']).reset_index(drop=True)

# Create spread dataframe for PICNIC_BASKET1 with continuous index
df_basket_1['continuous_index'] = range(len(df_basket_1))
df_synthetic_1['continuous_index'] = range(len(df_synthetic_1))

In [ ]:
#STRATEGY 1 With Just spread


from tqdm import tqdm
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import itertools

# 1. Load data from all days
days = [-1, 0, 1]  # Days as requested
df_all = []

for day in days:
    df_day = pd.read_csv(f"prices_round_2_day_{day}.csv", sep=";", header=0)
    df_day['day'] = day  # Add day identifier
    df_all.append(df_day)

# Concatenate all days
df = pd.concat(df_all, ignore_index=True)
print(f"Loaded data from all days. Total rows: {len(df)}")

# 2. Process all the data
df_croissant = df[df['product'] == 'CROISSANTS'].copy()
df_jam = df[df['product'] == 'JAMS'].copy()
df_djembe = df[df['product'] == 'DJEMBES'].copy()
df_basket_1 = df[df['product'] == 'PICNIC_BASKET1'].copy()
df_basket_2 = df[df['product'] == 'PICNIC_BASKET2'].copy()

# Calculate smart weighted mid price for each product
df_croissant['swmid'] = (df_croissant['bid_price_1'] * df_croissant['ask_volume_1'] + df_croissant['ask_price_1'] * df_croissant['bid_volume_1']) / (df_croissant['ask_volume_1'] + df_croissant['bid_volume_1'])
df_jam['swmid'] = (df_jam['bid_price_1'] * df_jam['ask_volume_1'] + df_jam['ask_price_1'] * df_jam['bid_volume_1']) / (df_jam['ask_volume_1'] + df_jam['bid_volume_1'])
df_djembe['swmid'] = (df_djembe['bid_price_1'] * df_djembe['ask_volume_1'] + df_djembe['ask_price_1'] * df_djembe['bid_volume_1']) / (df_djembe['ask_volume_1'] + df_djembe['bid_volume_1'])
df_basket_1['swmid'] = (df_basket_1['bid_price_1'] * df_basket_1['ask_volume_1'] + df_basket_1['ask_price_1'] * df_basket_1['bid_volume_1']) / (df_basket_1['ask_volume_1'] + df_basket_1['bid_volume_1'])
df_basket_2['swmid'] = (df_basket_2['bid_price_1'] * df_basket_2['ask_volume_1'] + df_basket_2['ask_price_1'] * df_basket_2['bid_volume_1']) / (df_basket_2['ask_volume_1'] + df_basket_2['bid_volume_1'])

# Sort by day and timestamp to ensure chronological order
df_croissant = df_croissant.sort_values(['day', 'timestamp']).reset_index(drop=True)
df_jam = df_jam.sort_values(['day', 'timestamp']).reset_index(drop=True)
df_djembe = df_djembe.sort_values(['day', 'timestamp']).reset_index(drop=True)
df_basket_1 = df_basket_1.sort_values(['day', 'timestamp']).reset_index(drop=True)
df_basket_2 = df_basket_2.sort_values(['day', 'timestamp']).reset_index(drop=True)

# Create synthetic prices for PICNIC_BASKET1 (6 CROISSANTS, 3 JAMS, 1 DJEMBE)
df_synthetic_1 = pd.DataFrame({
    'timestamp': df_croissant['timestamp'].to_numpy(),
    'day': df_croissant['day'].to_numpy(),
    'bid_price_1': df_croissant['bid_price_1'].to_numpy() * 6 + df_jam['bid_price_1'].to_numpy() * 3 + df_djembe['bid_price_1'].to_numpy(),
    'bid_volume_1': np.min(np.array([
        df_croissant['bid_volume_1'].to_numpy(), 
        df_jam['bid_volume_1'].to_numpy(), 
        df_djembe['bid_volume_1'].to_numpy()
    ]), axis=0),
    'ask_price_1': df_croissant['ask_price_1'].to_numpy() * 6 + df_jam['ask_price_1'].to_numpy() * 3 + df_djembe['ask_price_1'].to_numpy(),
    'ask_volume_1': np.min(np.array([
        df_croissant['ask_volume_1'].to_numpy(), 
        df_jam['ask_volume_1'].to_numpy(), 
        df_djembe['ask_volume_1'].to_numpy()
    ]), axis=0),
    'mid_price': df_croissant['mid_price'].to_numpy() * 6 + df_jam['mid_price'].to_numpy() * 3 + df_djembe['mid_price'].to_numpy(),
    'swmid': df_croissant['swmid'].to_numpy() * 6 + df_jam['swmid'].to_numpy() * 3 + df_djembe['swmid'].to_numpy()
})

# Create synthetic prices for PICNIC_BASKET2 (4 CROISSANTS, 2 JAMS)
df_synthetic_2 = pd.DataFrame({
    'timestamp': df_croissant['timestamp'].to_numpy(),
    'day': df_croissant['day'].to_numpy(),
    'bid_price_1': df_croissant['bid_price_1'].to_numpy() * 4 + df_jam['bid_price_1'].to_numpy() * 2,
    'bid_volume_1': np.min(np.array([
        df_croissant['bid_volume_1'].to_numpy(), 
        df_jam['bid_volume_1'].to_numpy()
    ]), axis=0),
    'ask_price_1': df_croissant['ask_price_1'].to_numpy() * 4 + df_jam['ask_price_1'].to_numpy() * 2,
    'ask_volume_1': np.min(np.array([
        df_croissant['ask_volume_1'].to_numpy(), 
        df_jam['ask_volume_1'].to_numpy()
    ]), axis=0),
    'mid_price': df_croissant['mid_price'].to_numpy() * 4 + df_jam['mid_price'].to_numpy() * 2,
    'swmid': df_croissant['swmid'].to_numpy() * 4 + df_jam['swmid'].to_numpy() * 2
})

# Add continuous index
df_basket_1['continuous_index'] = range(len(df_basket_1))
df_synthetic_1['continuous_index'] = range(len(df_synthetic_1))

# Create spread dataframe 
spread = pd.DataFrame({
    'timestamp': df_synthetic_1['timestamp'],
    'day': df_synthetic_1['day'],
    'continuous_index': df_synthetic_1['continuous_index'],
    'spread': df_basket_1['swmid'].to_numpy() - df_synthetic_1['swmid'].to_numpy()
})

# Visualize the spread 
plt.figure(figsize=(14, 7))
plt.plot(spread['continuous_index'], spread['spread'], color='purple', alpha=0.7)

mean_spread = spread['spread'].mean()
plt.axhline(y=mean_spread, color='blue', linestyle='--', label=f'Mean: {mean_spread:.2f}')
plt.title("Picnic Basket 1 - Synthetic Spread")
plt.xlabel("Trading Period")
plt.ylabel("Spread")
plt.legend()
plt.grid(True)
plt.show()

# Calculate mean spread
mean_spread = spread['spread'].mean()
print(f"Mean spread value: {mean_spread}")

def cross_spread(cash, quantity):
    # Crossing cost - adjust the 10 value if needed for your market
    return cash - abs(quantity) * 10

# Function to calculate max drawdown
def calculate_max_drawdown(pnl_hist):
    running_max = np.maximum.accumulate(pnl_hist)
    drawdowns = running_max - pnl_hist
    max_drawdown = np.max(drawdowns)
    return max_drawdown

def backtest(thresh, target_position, std_window, verbose=False):
    cash = 0
    position = 0
    pnl_hist = []
    position_hist = []
    cash_hist = []
    day_markers = []
    order_count = 0  # Track number of orders placed
    
    # Calculate standard deviation with specified window
    spread[f'std{std_window}'] = spread['spread'].rolling(window=std_window).std()
    z_score = (spread['spread'].to_numpy() - mean_spread) / spread[f'std{std_window}'].to_numpy()
    
    # Create market data with z-scores
    spread_market = pd.DataFrame({
        'timestamp': df_synthetic_1['timestamp'].to_numpy(),
        'continuous_index': df_synthetic_1['continuous_index'].to_numpy(),
        'swmid': df_basket_1['swmid'].to_numpy() - df_synthetic_1['swmid'].to_numpy(),
        'spread_z': z_score
    })
    
    for index, row in spread_market.iterrows():
        if index == 0:
            continue
            
        # Mark day transitions for visualization
        if index > 1 and spread_market.iloc[index-1]['timestamp'] > row['timestamp']:
            day_markers.append(index)
            
        swmid = row['swmid']
        
        # Trade based on z-score only (original strategy)
        if not np.isnan(row['spread_z']):
            # Check for sell signal
            if row['spread_z'] > thresh and position != -target_position:
                quantity = -target_position - position
                cash -= (-target_position - position) * swmid
                cash = cross_spread(cash, quantity)
                position = -target_position
                order_count += 1  # Count one order placed
                
                if verbose:
                    print(f"SELL {quantity} AT PRICE {swmid} AT TIME {row['timestamp']}")
            
            # Check for buy signal
            if row['spread_z'] < -thresh and position != target_position:
                quantity = target_position - position
                cash -= (target_position - position) * swmid
                cash = cross_spread(cash, quantity)
                position = target_position
                order_count += 1  # Count one order placed
                
                if verbose:
                    print(f"BUY {quantity} AT PRICE {swmid} AT TIME {row['timestamp']}")
        
        position_hist.append(position)
        cash_hist.append(cash)
        pnl_hist.append(cash + position * swmid)
        
    # Calculate max drawdown
    max_drawdown = calculate_max_drawdown(pnl_hist)
    
    if verbose:
        print(f"PNL: {pnl_hist[-1]}")
        print(f"Max Drawdown: {max_drawdown}")
        print(f"Number of orders placed: {order_count}")
        
    return pnl_hist, day_markers, position_hist, max_drawdown, order_count

# Set optimization parameters
position_opt = [60]  # Use position limit for PICNIC_BASKET1
thresh_opt = [1,1.3,1.6,1.9,2.1]  # Thresholds
std_window_opt = [10,15, 20,25, 30,25, 40,45, 50]  # Standard deviation windows
opt = []

# Run grid search for optimal parameters
print(f"Running grid search with {len(thresh_opt) * len(std_window_opt) * len(position_opt)} parameter combinations...")

for thresh, std_window, position in tqdm(list(itertools.product(thresh_opt, std_window_opt, position_opt))):
    try:
        pnl, _, _, max_drawdown, order_count = backtest(thresh, position, std_window)
        opt.append({
            "thresh": thresh, 
            "position": position, 
            "std_window": std_window,
            "pnl": pnl,
            "final_pnl": pnl[-1],
            "max_drawdown": max_drawdown,
            "order_count": order_count
        })
        print("="*80)
        print(f"Thresh: {thresh}, Position: {position}, Std Window: {std_window}")
        print(f"PnL: {pnl[-1]}, Max Drawdown: {max_drawdown}, Orders: {order_count}")
        print("="*80)
    except Exception as e:
        print(f"Error with parameters: thresh={thresh}, position={position}, std_window={std_window}")
        print(f"Error message: {str(e)}")

# Sort results by PnL (descending)
sorted_results = sorted(opt, key=lambda x: x["final_pnl"], reverse=True)
print("\nTop 5 Parameter Sets for PICNIC_BASKET1:")
print("="*80)
for i, result in enumerate(sorted_results[:5]):
    print(f"Rank {i+1}:")
    print(f"  Threshold: {result['thresh']}")
    print(f"  Position: {result['position']}")
    print(f"  Std Window: {result['std_window']}")
    print(f"  PnL: {result['final_pnl']}")
    print(f"  Max Drawdown: {result['max_drawdown']}")
    print(f"  Order Count: {result['order_count']}")
    print("-"*40)

# Find best parameters
best_pnl = -float('inf')
best_params = None
for result in opt:
    if result["final_pnl"] > best_pnl:
        best_pnl = result["final_pnl"]
        best_params = {
            "thresh": result["thresh"],
            "position": result["position"],
            "std_window": result["std_window"]
        }

print(f"Best parameters: {best_params}")
print(f"Best PnL: {best_pnl}")

# Plot top 3 parameter sets with position overlay
top3_results = sorted_results[:3]

fig, axes = plt.subplots(3, 2, figsize=(18, 15))
fig.suptitle("Top 3 Parameter Sets for PICNIC_BASKET1 (Original Strategy)", fontsize=16)

for i, result in enumerate(top3_results):
    # Get day markers and positions for this parameter set
    pnl, day_markers, positions, max_drawdown, order_count = backtest(
        result["thresh"], 
        result["position"], 
        result["std_window"]
    )
    
    # Plot PnL
    axes[i, 0].plot(pnl, color='blue')
    for marker in day_markers:
        axes[i, 0].axvline(x=marker, color='gray', linestyle='--')
    
    title = (f"Rank {i+1}: Thresh={result['thresh']}, Pos={result['position']}, " + 
             f"StdWin={result['std_window']}")
    axes[i, 0].set_title(title)
    axes[i, 0].set_ylabel("PnL")
    axes[i, 0].grid(True)
    
    # Annotate final PnL and drawdown
    axes[i, 0].annotate(f"Final PnL: {result['final_pnl']:.2f}\nMax DD: {result['max_drawdown']:.2f}\nOrders: {result['order_count']}", 
                      xy=(0.02, 0.85), xycoords='axes fraction',
                      bbox=dict(boxstyle="round,pad=0.3", fc="white", ec="gray", alpha=0.8))
    
    # Plot positions
    axes[i, 1].plot(positions, color='green')
    axes[i, 1].set_ylabel("Position")
    axes[i, 1].set_title(f"Positions for Parameter Set {i+1}")
    axes[i, 1].grid(True)

plt.tight_layout(rect=[0, 0, 1, 0.96])  # Adjust for suptitle
plt.show()

# Run a verbose backtest with best parameters to show detailed trade information
print("Running backtest with best parameters and verbose output:")
backtest(best_params["thresh"], best_params["position"], best_params["std_window"], verbose=True)

# Run same analysis for PICNIC_BASKET2
print("\n" + "="*80)
print("Running analysis for PICNIC_BASKET2")
print("="*80)

# Create initial spread for basket 2
spread_basket2 = pd.DataFrame({
    'timestamp': df_synthetic_2['timestamp'],
    'day': df_synthetic_2['day'],
    'spread': df_basket_2['swmid'].to_numpy() - df_synthetic_2['swmid'].to_numpy()
})

# Calculate mean spread
mean_spread_basket2 = spread_basket2['spread'].mean()
print(f"Mean spread value for PICNIC_BASKET2: {mean_spread_basket2}")

# Same backtest function, adapted for basket 2
def backtest_basket2(thresh, target_position, std_window, verbose=False):
    cash = 0
    position = 0
    pnl_hist = []
    position_hist = []
    cash_hist = []
    order_count = 0  # Track number of orders placed
    
    # Calculate standard deviation and z-score
    spread_basket2[f'std{std_window}'] = spread_basket2['spread'].rolling(window=std_window).std()
    z_score = (spread_basket2['spread'].to_numpy() - mean_spread_basket2) / spread_basket2[f'std{std_window}'].to_numpy()
    
    spread_market_basket2 = pd.DataFrame({
        'timestamp': df_synthetic_2['timestamp'].to_numpy(),
        'swmid': df_basket_2['swmid'].to_numpy() - df_synthetic_2['swmid'].to_numpy(),
        'spread_z': z_score
    })
    
    for index, row in spread_market_basket2.iterrows():
        if index == 0:
            continue
        swmid = row['swmid']
        
        # Trade based on z-score only (original strategy)
        if not np.isnan(row['spread_z']):
            if row['spread_z'] > thresh and position != -target_position:
                quantity = -target_position - position
                cash -= (-target_position - position) * swmid
                cash = cross_spread(cash, quantity)
                position = -target_position
                order_count += 1  # Count one order placed
                
                if verbose:
                    print(f"SELL {quantity} AT PRICE {swmid} AT TIME {row['timestamp']}")
            
            if row['spread_z'] < -thresh and position != target_position:
                quantity = target_position - position
                cash -= (target_position - position) * swmid
                cash = cross_spread(cash, quantity)
                position = target_position
                order_count += 1  # Count one order placed
                
                if verbose:
                    print(f"BUY {quantity} AT PRICE {swmid} AT TIME {row['timestamp']}")
    
        position_hist.append(position)
        cash_hist.append(cash)
        pnl_hist.append(cash + position * swmid)
    
    # Calculate max drawdown
    max_drawdown = calculate_max_drawdown(pnl_hist)
        
    if verbose:
        print(f"PNL: {pnl_hist[-1]}")
        print(f"Max Drawdown: {max_drawdown}")
        print(f"Number of orders placed: {order_count}")
        
    return pnl_hist, max_drawdown, position_hist, order_count

# Optimization for basket 2
position_opt_basket2 = [100]  # PICNIC_BASKET2 position limit is 100
opt_basket2 = []

print(f"Running grid search for Basket 2 with {len(thresh_opt) * len(std_window_opt) * len(position_opt_basket2)} parameter combinations...")

for thresh, std_window, position in tqdm(list(itertools.product(thresh_opt, std_window_opt, position_opt_basket2))):
    try:
        pnl, max_drawdown, positions, order_count = backtest_basket2(thresh, position, std_window)
        opt_basket2.append({
            "thresh": thresh, 
            "position": position, 
            "std_window": std_window,
            "pnl": pnl,
            "final_pnl": pnl[-1],
            "max_drawdown": max_drawdown,
            "order_count": order_count
        })
        print("="*80)
        print(f"BASKET2 - Thresh: {thresh}, Position: {position}, Std Window: {std_window}")
        print(f"PnL: {pnl[-1]}, Max Drawdown: {max_drawdown}, Orders: {order_count}")
        print("="*80)
    except Exception as e:
        print(f"Error with parameters: thresh={thresh}, position={position}, std_window={std_window}")
        print(f"Error message: {str(e)}")

# Sort results by PnL (descending) for basket 2
sorted_results_basket2 = sorted(opt_basket2, key=lambda x: x["final_pnl"], reverse=True)
print("\nTop 5 Parameter Sets for PICNIC_BASKET2:")
print("="*80)
for i, result in enumerate(sorted_results_basket2[:5]):
    print(f"Rank {i+1}:")
    print(f"  Threshold: {result['thresh']}")
    print(f"  Position: {result['position']}")
    print(f"  Std Window: {result['std_window']}")
    print(f"  PnL: {result['final_pnl']}")
    print(f"  Max Drawdown: {result['max_drawdown']}")
    print(f"  Order Count: {result['order_count']}")
    print("-"*40)

# Find best parameters for basket 2
best_pnl_basket2 = -float('inf')
best_params_basket2 = None
for result in opt_basket2:
    if result["final_pnl"] > best_pnl_basket2:
        best_pnl_basket2 = result["final_pnl"]
        best_params_basket2 = {
            "thresh": result["thresh"],
            "position": result["position"],
            "std_window": result["std_window"]
        }

print(f"Best parameters for PICNIC_BASKET2: {best_params_basket2}")
print(f"Best PnL for PICNIC_BASKET2: {best_pnl_basket2}")

# Plot top 3 parameter sets for PICNIC_BASKET2
top3_results_basket2 = sorted_results_basket2[:3]

fig, axes = plt.subplots(3, 2, figsize=(18, 15))
fig.suptitle("Top 3 Parameter Sets for PICNIC_BASKET2 (Original Strategy)", fontsize=16)

for i, result in enumerate(top3_results_basket2):
    # Get positions for this parameter set
    pnl, max_drawdown, positions, order_count = backtest_basket2(
        result["thresh"], 
        result["position"], 
        result["std_window"]
    )
    
    # Plot PnL
    axes[i, 0].plot(pnl, color='blue')
    
    title = (f"Rank {i+1}: Thresh={result['thresh']}, Pos={result['position']}, " + 
             f"StdWin={result['std_window']}")
    axes[i, 0].set_title(title)
    axes[i, 0].set_ylabel("PnL")
    axes[i, 0].grid(True)
    
    # Annotate final PnL
    axes[i, 0].annotate(f"Final PnL: {result['final_pnl']:.2f}\nMax DD: {result['max_drawdown']:.2f}\nOrders: {result['order_count']}", 
                      xy=(0.02, 0.85), xycoords='axes fraction',
                      bbox=dict(boxstyle="round,pad=0.3", fc="white", ec="gray", alpha=0.8))
    
    # Plot positions
    axes[i, 1].plot(positions, color='green')
    axes[i, 1].set_ylabel("Position")
    axes[i, 1].set_title(f"Positions for Parameter Set {i+1}")
    axes[i, 1].grid(True)

plt.tight_layout(rect=[0, 0, 1, 0.96])  # Adjust for suptitle
plt.show()

# Run a verbose backtest with best parameters for basket 2
print("Running verbose backtest with best parameters for PICNIC_BASKET2:")
backtest_basket2(best_params_basket2["thresh"], best_params_basket2["position"], 
                best_params_basket2["std_window"], verbose=True)

# Compare the best performance between original and MA-filtered strategies
print("\n" + "="*80)
print("COMPARISON: Original vs MA-Filtered Strategy")
print("="*80)

print("PICNIC_BASKET1:")
print(f"  Original Strategy - Best PnL: {best_pnl}")
print(f"  Parameters: Threshold={best_params['thresh']}, StdWindow={best_params['std_window']}")
print("\nPICNIC_BASKET2:")
print(f"  Original Strategy - Best PnL: {best_pnl_basket2}")
print(f"  Parameters: Threshold={best_params_basket2['thresh']}, StdWindow={best_params_basket2['std_window']}")

In [ ]:
from tqdm import tqdm
import pandas as pd
import numpy as np
import itertools

def backtest_single_day(thresh, target_position, std_window, verbose=False):
    # 2. Process all the data
    df_croissant = df[df['product'] == 'CROISSANTS'].copy()
    df_jam = df[df['product'] == 'JAMS'].copy()
    df_djembe = df[df['product'] == 'DJEMBES'].copy()
    df_basket_1 = df[df['product'] == 'PICNIC_BASKET1'].copy()

    # Calculate smart weighted mid price for each product
    def calculate_swmid(df):
        return (df['bid_price_1'] * df['ask_volume_1'] + df['ask_price_1'] * df['bid_volume_1']) / (df['ask_volume_1'] + df['bid_volume_1'])

    df_croissant['swmid'] = calculate_swmid(df_croissant)
    df_jam['swmid'] = calculate_swmid(df_jam)
    df_djembe['swmid'] = calculate_swmid(df_djembe)
    df_basket_1['swmid'] = calculate_swmid(df_basket_1)

    # Sort by timestamp to ensure chronological order
    df_croissant = df_croissant.sort_values('timestamp').reset_index(drop=True)
    df_jam = df_jam.sort_values('timestamp').reset_index(drop=True)
    df_djembe = df_djembe.sort_values('timestamp').reset_index(drop=True)
    df_basket_1 = df_basket_1.sort_values('timestamp').reset_index(drop=True)

    # Create synthetic prices for PICNIC_BASKET1 (6 CROISSANTS, 3 JAMS, 1 DJEMBE)
    df_synthetic_1 = pd.DataFrame({
        'timestamp': df_croissant['timestamp'].to_numpy(),
        'bid_price_1': (df_croissant['bid_price_1'].to_numpy() * 6 + 
                        df_jam['bid_price_1'].to_numpy() * 3 + 
                        df_djembe['bid_price_1'].to_numpy()),
        'bid_volume_1': np.min(np.array([
            df_croissant['bid_volume_1'].to_numpy(), 
            df_jam['bid_volume_1'].to_numpy(), 
            df_djembe['bid_volume_1'].to_numpy()
        ]), axis=0),
        'ask_price_1': (df_croissant['ask_price_1'].to_numpy() * 6 + 
                        df_jam['ask_price_1'].to_numpy() * 3 + 
                        df_djembe['ask_price_1'].to_numpy()),
        'ask_volume_1': np.min(np.array([
            df_croissant['ask_volume_1'].to_numpy(), 
            df_jam['ask_volume_1'].to_numpy(), 
            df_djembe['ask_volume_1'].to_numpy()
        ]), axis=0),
        'swmid': (df_croissant['swmid'].to_numpy() * 6 + 
                  df_jam['swmid'].to_numpy() * 3 + 
                  df_djembe['swmid'].to_numpy())
    })

    # Create spread dataframe 
    spread = pd.DataFrame({
        'timestamp': df_synthetic_1['timestamp'],
        'spread': df_basket_1['swmid'].to_numpy() - df_synthetic_1['swmid'].to_numpy()
    })

    # Calculate mean spread
    mean_spread = spread['spread'].mean()

    def cross_spread(cash, quantity):
        # Crossing cost - adjust the 10 value if needed for your market
        return cash - abs(quantity) * 10

    def calculate_max_drawdown(pnl_hist):
        running_max = np.maximum.accumulate(pnl_hist)
        drawdowns = running_max - pnl_hist
        max_drawdown = np.max(drawdowns)
        return max_drawdown

    # Calculate standard deviation with specified window
    spread[f'std{std_window}'] = spread['spread'].rolling(window=std_window).std()
    z_score = (spread['spread'].to_numpy() - mean_spread) / spread[f'std{std_window}'].to_numpy()

    # Create market data with z-scores
    spread_market = pd.DataFrame({
        'timestamp': df_synthetic_1['timestamp'].to_numpy(),
        'swmid': df_basket_1['swmid'].to_numpy() - df_synthetic_1['swmid'].to_numpy(),
        'spread_z': z_score
    })

    # Backtest logic
    cash = 0
    position = 0
    pnl_hist = []
    position_hist = []
    cash_hist = []
    order_count = 0

    for index, row in spread_market.iterrows():
        if index == 0:
            continue

        swmid = row['swmid']

        # Trade based on z-score only
        if not np.isnan(row['spread_z']):
            # Check for sell signal
            if row['spread_z'] > thresh and position != -target_position:
                quantity = -target_position - position
                cash -= (-target_position - position) * swmid
                cash = cross_spread(cash, quantity)
                position = -target_position
                order_count += 1

                if verbose:
                    print(f"SELL {quantity} AT PRICE {swmid} AT TIME {row['timestamp']}")

            # Check for buy signal
            if row['spread_z'] < -thresh and position != target_position:
                quantity = target_position - position
                cash -= (target_position - position) * swmid
                cash = cross_spread(cash, quantity)
                position = target_position
                order_count += 1

                if verbose:
                    print(f"BUY {quantity} AT PRICE {swmid} AT TIME {row['timestamp']}")

        position_hist.append(position)
        cash_hist.append(cash)
        pnl_hist.append(cash + position * swmid)

    # Calculate max drawdown
    max_drawdown = calculate_max_drawdown(pnl_hist)

    if verbose:
        print(f"PNL: {pnl_hist[-1]}")
        print(f"Max Drawdown: {max_drawdown}")
        print(f"Number of orders placed: {order_count}")

    return pnl_hist[-1], max_drawdown, order_count

# Optimization parameters
position_opt = [60]  # Position limit for PICNIC_BASKET1
thresh_opt = [1,1.5, 2,2.5, 3,3.5]  # Thresholds
std_window_opt = [10, 15, 20, 25, 30, 40, 50]  # Standard deviation windows

# Run optimization
results = []

for thresh, std_window, position in tqdm(list(itertools.product(thresh_opt, std_window_opt, position_opt))):
    try:
        pnl, max_drawdown, order_count = backtest_single_day(thresh, position, std_window)
        results.append({
            "thresh": thresh, 
            "position": position, 
            "std_window": std_window,
            "pnl": pnl,
            "max_drawdown": max_drawdown,
            "order_count": order_count
        })
    except Exception as e:
        print(f"Error with parameters: thresh={thresh}, position={position}, std_window={std_window}")
        print(f"Error message: {str(e)}")

# Sort results
sorted_results = sorted(results, key=lambda x: x["pnl"], reverse=True)

# Print top 5 results
print("\nTop 5 Parameter Sets:")
for i, result in enumerate(sorted_results[:5]):
    print(f"Rank {i+1}:")
    print(f"  Threshold: {result['thresh']}")
    print(f"  Position: {result['position']}")
    print(f"  Std Window: {result['std_window']}")
    print(f"  PnL: {result['pnl']}")
    print(f"  Max Drawdown: {result['max_drawdown']}")
    print(f"  Order Count: {result['order_count']}")
    print("-"*40)

# Find best parameters
best_result = sorted_results[0]
print("\nBest Parameters:")
print(f"  Threshold: {best_result['thresh']}")
print(f"  Position: {best_result['position']}")
print(f"  Std Window: {best_result['std_window']}")

In [ ]:
#STRATEGY 3 ONLY WHEN JAM AND CROISSANT MOVES
from tqdm import tqdm
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import itertools

# 1. Load data from all days
days = [-1, 0, 1]  # Days as requested
df_all = []

for day in days:
    df_day = pd.read_csv(f"prices_round_2_day_{day}.csv", sep=";", header=0)
    df_day['day'] = day  # Add day identifier
    df_all.append(df_day)

# Concatenate all days
df = pd.concat(df_all, ignore_index=True)
print(f"Loaded data from all days. Total rows: {len(df)}")

# 2. Process all the data
df_croissant = df[df['product'] == 'CROISSANTS'].copy()
df_jam = df[df['product'] == 'JAMS'].copy()
df_djembe = df[df['product'] == 'DJEMBES'].copy()
df_basket_1 = df[df['product'] == 'PICNIC_BASKET1'].copy()
df_basket_2 = df[df['product'] == 'PICNIC_BASKET2'].copy()

# Calculate smart weighted mid price for each product
df_croissant['swmid'] = (df_croissant['bid_price_1'] * df_croissant['ask_volume_1'] + df_croissant['ask_price_1'] * df_croissant['bid_volume_1']) / (df_croissant['ask_volume_1'] + df_croissant['bid_volume_1'])
df_jam['swmid'] = (df_jam['bid_price_1'] * df_jam['ask_volume_1'] + df_jam['ask_price_1'] * df_jam['bid_volume_1']) / (df_jam['ask_volume_1'] + df_jam['bid_volume_1'])
df_djembe['swmid'] = (df_djembe['bid_price_1'] * df_djembe['ask_volume_1'] + df_djembe['ask_price_1'] * df_djembe['bid_volume_1']) / (df_djembe['ask_volume_1'] + df_djembe['bid_volume_1'])
df_basket_1['swmid'] = (df_basket_1['bid_price_1'] * df_basket_1['ask_volume_1'] + df_basket_1['ask_price_1'] * df_basket_1['bid_volume_1']) / (df_basket_1['ask_volume_1'] + df_basket_1['bid_volume_1'])
df_basket_2['swmid'] = (df_basket_2['bid_price_1'] * df_basket_2['ask_volume_1'] + df_basket_2['ask_price_1'] * df_basket_2['bid_volume_1']) / (df_basket_2['ask_volume_1'] + df_basket_2['bid_volume_1'])

# Sort by day and timestamp to ensure chronological order
df_croissant = df_croissant.sort_values(['day', 'timestamp']).reset_index(drop=True)
df_jam = df_jam.sort_values(['day', 'timestamp']).reset_index(drop=True)
df_djembe = df_djembe.sort_values(['day', 'timestamp']).reset_index(drop=True)
df_basket_1 = df_basket_1.sort_values(['day', 'timestamp']).reset_index(drop=True)
df_basket_2 = df_basket_2.sort_values(['day', 'timestamp']).reset_index(drop=True)

# Function to calculate moving averages and directions for a given window size
def calculate_asset_directions(ma_window):
    # Calculate moving averages for each asset
    df_croissant['ma'] = df_croissant['swmid'].rolling(window=ma_window).mean()
    df_jam['ma'] = df_jam['swmid'].rolling(window=ma_window).mean()
    df_djembe['ma'] = df_djembe['swmid'].rolling(window=ma_window).mean()
    
    # Calculate direction for each asset (1: up, -1: down, 0: flat)
    df_croissant['direction'] = np.sign(df_croissant['swmid'] - df_croissant['ma'].shift(1))
    df_jam['direction'] = np.sign(df_jam['swmid'] - df_jam['ma'].shift(1))
    df_djembe['direction'] = np.sign(df_djembe['swmid'] - df_djembe['ma'].shift(1))
    
    return df_croissant['direction'], df_jam['direction'], df_djembe['direction']

# Initial direction calculation with default window
default_ma_window = 10  # Default for initial plotting
croissant_dir, jam_dir, djembe_dir = calculate_asset_directions(default_ma_window)

# Create synthetic prices for PICNIC_BASKET1 (6 CROISSANTS, 3 JAMS, 1 DJEMBE)
df_synthetic_1 = pd.DataFrame({
    'timestamp': df_croissant['timestamp'].to_numpy(),
    'day': df_croissant['day'].to_numpy(),
    'bid_price_1': df_croissant['bid_price_1'].to_numpy() * 6 + df_jam['bid_price_1'].to_numpy() * 3 + df_djembe['bid_price_1'].to_numpy(),
    'bid_volume_1': np.min(np.array([
        df_croissant['bid_volume_1'].to_numpy(), 
        df_jam['bid_volume_1'].to_numpy(), 
        df_djembe['bid_volume_1'].to_numpy()
    ]), axis=0),
    'ask_price_1': df_croissant['ask_price_1'].to_numpy() * 6 + df_jam['ask_price_1'].to_numpy() * 3 + df_djembe['ask_price_1'].to_numpy(),
    'ask_volume_1': np.min(np.array([
        df_croissant['ask_volume_1'].to_numpy(), 
        df_jam['ask_volume_1'].to_numpy(), 
        df_djembe['ask_volume_1'].to_numpy()
    ]), axis=0),
    'mid_price': df_croissant['mid_price'].to_numpy() * 6 + df_jam['mid_price'].to_numpy() * 3 + df_djembe['mid_price'].to_numpy(),
    'swmid': df_croissant['swmid'].to_numpy() * 6 + df_jam['swmid'].to_numpy() * 3 + df_djembe['swmid'].to_numpy()
})

# Create synthetic prices for PICNIC_BASKET2 (4 CROISSANTS, 2 JAMS)
df_synthetic_2 = pd.DataFrame({
    'timestamp': df_croissant['timestamp'].to_numpy(),
    'day': df_croissant['day'].to_numpy(),
    'bid_price_1': df_croissant['bid_price_1'].to_numpy() * 4 + df_jam['bid_price_1'].to_numpy() * 2,
    'bid_volume_1': np.min(np.array([
        df_croissant['bid_volume_1'].to_numpy(), 
        df_jam['bid_volume_1'].to_numpy()
    ]), axis=0),
    'ask_price_1': df_croissant['ask_price_1'].to_numpy() * 4 + df_jam['ask_price_1'].to_numpy() * 2,
    'ask_volume_1': np.min(np.array([
        df_croissant['ask_volume_1'].to_numpy(), 
        df_jam['ask_volume_1'].to_numpy()
    ]), axis=0),
    'mid_price': df_croissant['mid_price'].to_numpy() * 4 + df_jam['mid_price'].to_numpy() * 2,
    'swmid': df_croissant['swmid'].to_numpy() * 4 + df_jam['swmid'].to_numpy() * 2
})

# Add continuous index
df_basket_1['continuous_index'] = range(len(df_basket_1))
df_synthetic_1['continuous_index'] = range(len(df_synthetic_1))

# Create spread dataframe 
spread = pd.DataFrame({
    'timestamp': df_synthetic_1['timestamp'],
    'day': df_synthetic_1['day'],
    'continuous_index': df_synthetic_1['continuous_index'],
    'spread': df_basket_1['swmid'].to_numpy() - df_synthetic_1['swmid'].to_numpy()
})

# Function to update the spread dataframe with direction information - modified to care only about JAMS and CROISSANTS
def update_spread_directions(spread, croissant_dir, jam_dir, djembe_dir):
    spread['croissant_direction'] = croissant_dir.to_numpy()
    spread['jam_direction'] = jam_dir.to_numpy()
    spread['djembe_direction'] = djembe_dir.to_numpy()
    
    # Add a column indicating if JAM and CROISSANT move in the same direction
    spread['jc_same_direction'] = ((spread['croissant_direction'] == spread['jam_direction']) & 
                                   (spread['croissant_direction'] != 0))  # Exclude flat movements
    return spread

# Update with initial directions for visualization
spread = update_spread_directions(spread, croissant_dir, jam_dir, djembe_dir)

# Visualize the spread with direction indicators
plt.figure(figsize=(14, 7))
plt.plot(spread['continuous_index'], spread['spread'], color='purple', alpha=0.7)

# Highlight periods where JAM and CROISSANT move in the same direction
same_dir_indices = spread[spread['jc_same_direction']].continuous_index
plt.scatter(same_dir_indices, spread.loc[same_dir_indices, 'spread'], 
           color='green', alpha=0.5, label='JAM and CROISSANT same direction')

mean_spread = spread['spread'].mean()
plt.axhline(y=mean_spread, color='blue', linestyle='--', label=f'Mean: {mean_spread:.2f}')
plt.title(f"Picnic Basket 1 - Synthetic Spread with JAM & CROISSANT Direction (MA Window: {default_ma_window})")
plt.xlabel("Trading Period")
plt.ylabel("Spread")
plt.legend()
plt.grid(True)
plt.show()

# Calculate mean spread
mean_spread = spread['spread'].mean()
print(f"Mean spread value: {mean_spread}")

def cross_spread(cash, quantity):
    # Crossing cost - adjust the 10 value if needed for your market
    return cash - abs(quantity) * 10

# Function to calculate max drawdown
def calculate_max_drawdown(pnl_hist):
    running_max = np.maximum.accumulate(pnl_hist)
    drawdowns = running_max - pnl_hist
    max_drawdown = np.max(drawdowns)
    return max_drawdown

def backtest(thresh, target_position, std_window, ma_window, verbose=False):
    cash = 0
    position = 0
    pnl_hist = []
    position_hist = []
    cash_hist = []
    day_markers = []
    order_count = 0  # Track number of orders placed
    
    # Recalculate asset directions for this MA window
    croissant_dir, jam_dir, djembe_dir = calculate_asset_directions(ma_window)
    # Update spread with these directions - focus on JAM and CROISSANT
    updated_spread = update_spread_directions(spread.copy(), croissant_dir, jam_dir, djembe_dir)
    
    # Calculate standard deviation with specified window
    updated_spread[f'std{std_window}'] = updated_spread['spread'].rolling(window=std_window).std()
    z_score = (updated_spread['spread'].to_numpy() - mean_spread) / updated_spread[f'std{std_window}'].to_numpy()
    
    # Create market data with z-scores and direction indicators
    spread_market = pd.DataFrame({
        'timestamp': df_synthetic_1['timestamp'].to_numpy(),
        'continuous_index': df_synthetic_1['continuous_index'].to_numpy(),
        'swmid': df_basket_1['swmid'].to_numpy() - df_synthetic_1['swmid'].to_numpy(),
        'spread_z': z_score,
        'jc_same_direction': updated_spread['jc_same_direction'].to_numpy()
    })
    
    for index, row in spread_market.iterrows():
        if index == 0:
            continue
            
        # Mark day transitions for visualization
        if index > 1 and spread_market.iloc[index-1]['timestamp'] > row['timestamp']:
            day_markers.append(index)
            
        swmid = row['swmid']
        
        # Only trade when JAM and CROISSANT are moving in the same direction
        if not np.isnan(row['spread_z']) and row['jc_same_direction']:
            # Check for sell signal
            if row['spread_z'] > thresh and position != -target_position:
                quantity = -target_position - position
                cash -= (-target_position - position) * swmid
                cash = cross_spread(cash, quantity)
                position = -target_position
                order_count += 1  # Count one order placed
                
                if verbose:
                    print(f"SELL {quantity} AT PRICE {swmid} AT TIME {row['timestamp']} (JAM & CROISSANT moving same direction)")
            
            # Check for buy signal
            if row['spread_z'] < -thresh and position != target_position:
                quantity = target_position - position
                cash -= (target_position - position) * swmid
                cash = cross_spread(cash, quantity)
                position = target_position
                order_count += 1  # Count one order placed
                
                if verbose:
                    print(f"BUY {quantity} AT PRICE {swmid} AT TIME {row['timestamp']} (JAM & CROISSANT moving same direction)")
        
        position_hist.append(position)
        cash_hist.append(cash)
        pnl_hist.append(cash + position * swmid)
        
    # Calculate max drawdown
    max_drawdown = calculate_max_drawdown(pnl_hist)
    
    if verbose:
        print(f"PNL: {pnl_hist[-1]}")
        print(f"Max Drawdown: {max_drawdown}")
        print(f"Number of trades where JAM & CROISSANT move in same direction: {sum(spread_market['jc_same_direction'])}")
        print(f"Number of orders placed: {order_count}")
        
    return pnl_hist, day_markers, position_hist, max_drawdown, order_count

# Set optimization parameters
position_opt = [60]  # Use position limit for PICNIC_BASKET1
thresh_opt = [1, 2, 3, 5, 7, 10,15,20,30]  # Reduced for computational efficiency
std_window_opt = [10,15, 20,25, 30, 40]  # Reduced for computational efficiency
ma_window_opt = [5, 10, 15, 20, 30]  # Moving average window options
opt = []

# Run grid search for optimal parameters
print(f"Running grid search with {len(thresh_opt) * len(std_window_opt) * len(position_opt) * len(ma_window_opt)} parameter combinations...")

for thresh, std_window, position, ma_window in tqdm(list(itertools.product(thresh_opt, std_window_opt, position_opt, ma_window_opt))):
    try:
        pnl, _, _, max_drawdown, order_count = backtest(thresh, position, std_window, ma_window)
        opt.append({
            "thresh": thresh, 
            "position": position, 
            "std_window": std_window, 
            "ma_window": ma_window, 
            "pnl": pnl,
            "final_pnl": pnl[-1],
            "max_drawdown": max_drawdown,
            "order_count": order_count
        })
        print("="*80)
        print(f"Thresh: {thresh}, Position: {position}, Std Window: {std_window}, MA Window: {ma_window}")
        print(f"PnL: {pnl[-1]}, Max Drawdown: {max_drawdown}, Orders: {order_count}")
        print("="*80)
    except Exception as e:
        print(f"Error with parameters: thresh={thresh}, position={position}, std_window={std_window}, ma_window={ma_window}")
        print(f"Error message: {str(e)}")

# Sort results by PnL (descending)
sorted_results = sorted(opt, key=lambda x: x["final_pnl"], reverse=True)
print("\nTop 5 Parameter Sets for PICNIC_BASKET1:")
print("="*80)
for i, result in enumerate(sorted_results[:5]):
    print(f"Rank {i+1}:")
    print(f"  Threshold: {result['thresh']}")
    print(f"  Position: {result['position']}")
    print(f"  Std Window: {result['std_window']}")
    print(f"  MA Window: {result['ma_window']}")
    print(f"  PnL: {result['final_pnl']}")
    print(f"  Max Drawdown: {result['max_drawdown']}")
    print(f"  Order Count: {result['order_count']}")
    print("-"*40)

# Find best parameters
best_pnl = -float('inf')
best_params = None
for result in opt:
    if result["final_pnl"] > best_pnl:
        best_pnl = result["final_pnl"]
        best_params = {
            "thresh": result["thresh"],
            "position": result["position"],
            "std_window": result["std_window"],
            "ma_window": result["ma_window"]
        }

print(f"Best parameters: {best_params}")
print(f"Best PnL: {best_pnl}")

# Plot top 3 parameter sets with position overlay
top3_results = sorted_results[:3]

fig, axes = plt.subplots(3, 2, figsize=(18, 15))
fig.suptitle("Top 3 Parameter Sets for PICNIC_BASKET1 (JAM & CROISSANT Direction Strategy)", fontsize=16)

for i, result in enumerate(top3_results):
    # Get day markers and positions for this parameter set
    pnl, day_markers, positions, max_drawdown, order_count = backtest(
        result["thresh"], 
        result["position"], 
        result["std_window"], 
        result["ma_window"]
    )
    
    # Plot PnL
    axes[i, 0].plot(pnl, color='blue')
    for marker in day_markers:
        axes[i, 0].axvline(x=marker, color='gray', linestyle='--')
    
    title = (f"Rank {i+1}: Thresh={result['thresh']}, Pos={result['position']}, " + 
             f"StdWin={result['std_window']}, MAWin={result['ma_window']}")
    axes[i, 0].set_title(title)
    axes[i, 0].set_ylabel("PnL")
    axes[i, 0].grid(True)
    
    # Annotate final PnL and drawdown
    axes[i, 0].annotate(f"Final PnL: {result['final_pnl']:.2f}\nMax DD: {result['max_drawdown']:.2f}\nOrders: {result['order_count']}", 
                      xy=(0.02, 0.85), xycoords='axes fraction',
                      bbox=dict(boxstyle="round,pad=0.3", fc="white", ec="gray", alpha=0.8))
    
    # Plot positions
    axes[i, 1].plot(positions, color='green')
    axes[i, 1].set_ylabel("Position")
    axes[i, 1].set_title(f"Positions for Parameter Set {i+1}")
    axes[i, 1].grid(True)

plt.tight_layout(rect=[0, 0, 1, 0.96])  # Adjust for suptitle
plt.show()

# Run a verbose backtest with best parameters to show detailed trade information
print("Running backtest with best parameters and verbose output:")
backtest(best_params["thresh"], best_params["position"], best_params["std_window"], best_params["ma_window"], verbose=True)

# Run same analysis for PICNIC_BASKET2
print("\n" + "="*80)
print("Running analysis for PICNIC_BASKET2")
print("="*80)

# Create initial spread for basket 2
spread_basket2 = pd.DataFrame({
    'timestamp': df_synthetic_2['timestamp'],
    'day': df_synthetic_2['day'],
    'spread': df_basket_2['swmid'].to_numpy() - df_synthetic_2['swmid'].to_numpy()
})

# Function to update the basket 2 spread dataframe with direction information
def update_basket2_directions(spread_basket2, croissant_dir, jam_dir):
    spread_basket2['croissant_direction'] = croissant_dir.to_numpy()
    spread_basket2['jam_direction'] = jam_dir.to_numpy()
    
    # Add indicator for when JAM and CROISSANT move in same direction
    spread_basket2['jc_same_direction'] = ((spread_basket2['croissant_direction'] == spread_basket2['jam_direction']) & 
                                          (spread_basket2['croissant_direction'] != 0))  # Exclude flat movements
    return spread_basket2

# Update with initial directions
spread_basket2 = update_basket2_directions(spread_basket2, croissant_dir, jam_dir)

# Calculate mean spread
mean_spread_basket2 = spread_basket2['spread'].mean()
print(f"Mean spread value for PICNIC_BASKET2: {mean_spread_basket2}")

# Same backtest function, adapted for basket 2
def backtest_basket2(thresh, target_position, std_window, ma_window, verbose=False):
    cash = 0
    position = 0
    pnl_hist = []
    position_hist = []
    cash_hist = []
    order_count = 0  # Track number of orders placed
    
    # Recalculate asset directions for this MA window
    croissant_dir, jam_dir, _ = calculate_asset_directions(ma_window)
    # Update spread with these directions
    updated_spread = update_basket2_directions(spread_basket2.copy(), croissant_dir, jam_dir)
    
    # Calculate standard deviation and z-score
    updated_spread[f'std{std_window}'] = updated_spread['spread'].rolling(window=std_window).std()
    z_score = (updated_spread['spread'].to_numpy() - mean_spread_basket2) / updated_spread[f'std{std_window}'].to_numpy()
    
    spread_market_basket2 = pd.DataFrame({
        'timestamp': df_synthetic_2['timestamp'].to_numpy(),
        'swmid': df_basket_2['swmid'].to_numpy() - df_synthetic_2['swmid'].to_numpy(),
        'spread_z': z_score,
        'jc_same_direction': updated_spread['jc_same_direction'].to_numpy()
    })
    
    for index, row in spread_market_basket2.iterrows():
        if index == 0:
            continue
        swmid = row['swmid']
        
        # Only trade when JAM and CROISSANT are moving in the same direction
        if not np.isnan(row['spread_z']) and row['jc_same_direction']:
            if row['spread_z'] > thresh and position != -target_position:
                quantity = -target_position - position
                cash -= (-target_position - position) * swmid
                cash = cross_spread(cash, quantity)
                position = -target_position
                order_count += 1  # Count one order placed
                
                if verbose:
                    print(f"SELL {quantity} AT PRICE {swmid} AT TIME {row['timestamp']} (JAM & CROISSANT moving same direction)")
            
            if row['spread_z'] < -thresh and position != target_position:
                quantity = target_position - position
                cash -= (target_position - position) * swmid
                cash = cross_spread(cash, quantity)
                position = target_position
                order_count += 1  # Count one order placed
                
                if verbose:
                    print(f"BUY {quantity} AT PRICE {swmid} AT TIME {row['timestamp']} (JAM & CROISSANT moving same direction)")
    
        position_hist.append(position)
        cash_hist.append(cash)
        pnl_hist.append(cash + position * swmid)
    
    # Calculate max drawdown
    max_drawdown = calculate_max_drawdown(pnl_hist)
        
    if verbose:
        print(f"PNL: {pnl_hist[-1]}")
        print(f"Max Drawdown: {max_drawdown}")
        print(f"Number of trades where JAM & CROISSANT move in same direction: {sum(spread_market_basket2['jc_same_direction'])}")
        print(f"Number of orders placed: {order_count}")
        
    return pnl_hist, max_drawdown, position_hist, order_count

# Optimization for basket 2 with MA window
position_opt_basket2 = [100]  # PICNIC_BASKET2 position limit is 100
opt_basket2 = []

print(f"Running grid search for Basket 2 with {len(thresh_opt) * len(std_window_opt) * len(position_opt_basket2) * len(ma_window_opt)} parameter combinations...")

for thresh, std_window, position, ma_window in tqdm(list(itertools.product(thresh_opt, std_window_opt, position_opt_basket2, ma_window_opt))):
    try:
        pnl, max_drawdown, positions, order_count = backtest_basket2(thresh, position, std_window, ma_window)
        opt_basket2.append({
            "thresh": thresh, 
            "position": position, 
            "std_window": std_window, 
            "ma_window": ma_window, 
            "pnl": pnl,
            "final_pnl": pnl[-1],
            "max_drawdown": max_drawdown,
            "order_count": order_count
        })
        print("="*80)
        print(f"BASKET2 - Thresh: {thresh}, Position: {position}, Std Window: {std_window}, MA Window: {ma_window}")
        print(f"PnL: {pnl[-1]}, Max Drawdown: {max_drawdown}, Orders: {order_count}")
        print("="*80)
    except Exception as e:
        print(f"Error with parameters: thresh={thresh}, position={position}, std_window={std_window}, ma_window={ma_window}")
        print(f"Error message: {str(e)}")

# Sort results by PnL (descending) for basket 2
sorted_results_basket2 = sorted(opt_basket2, key=lambda x: x["final_pnl"], reverse=True)
print("\nTop 5 Parameter Sets for PICNIC_BASKET2:")
print("="*80)
for i, result in enumerate(sorted_results_basket2[:5]):
    print(f"Rank {i+1}:")
    print(f"  Threshold: {result['thresh']}")
    print(f"  Position: {result['position']}")
    print(f"  Std Window: {result['std_window']}")
    print(f"  MA Window: {result['ma_window']}")
    print(f"  PnL: {result['final_pnl']}")
    print(f"  Max Drawdown: {result['max_drawdown']}")
    print(f"  Order Count: {result['order_count']}")
    print("-"*40)

# Find best parameters for basket 2
best_pnl_basket2 = -float('inf')
best_params_basket2 = None
for result in opt_basket2:
    if result["final_pnl"] > best_pnl_basket2:
        best_pnl_basket2 = result["final_pnl"]
        best_params_basket2 = {
            "thresh": result["thresh"],
            "position": result["position"],
            "std_window": result["std_window"],
            "ma_window": result["ma_window"]
        }

print(f"Best parameters for PICNIC_BASKET2: {best_params_basket2}")
print(f"Best PnL for PICNIC_BASKET2: {best_pnl_basket2}")

# Plot top 3 parameter sets for PICNIC_BASKET2
top3_results_basket2 = sorted_results_basket2[:3]

fig, axes = plt.subplots(3, 2, figsize=(18, 15))
fig.suptitle("Top 3 Parameter Sets for PICNIC_BASKET2 (JAM & CROISSANT Direction Strategy)", fontsize=16)

for i, result in enumerate(top3_results_basket2):
    # Get positions for this parameter set
    pnl, max_drawdown, positions, order_count = backtest_basket2(
        result["thresh"], 
        result["position"], 
        result["std_window"], 
        result["ma_window"]
    )
    
    # Plot PnL
    axes[i, 0].plot(pnl, color='blue')
    
    title = (f"Rank {i+1}: Thresh={result['thresh']}, Pos={result['position']}, " + 
             f"StdWin={result['std_window']}, MAWin={result['ma_window']}")
    axes[i, 0].set_title(title)
    axes[i, 0].set_ylabel("PnL")
    axes[i, 0].grid(True)
    
    # Annotate final PnL
    axes[i, 0].annotate(f"Final PnL: {result['final_pnl']:.2f}\nMax DD: {result['max_drawdown']:.2f}\nOrders: {result['order_count']}", 
                      xy=(0.02, 0.85), xycoords='axes fraction',
                      bbox=dict(boxstyle="round,pad=0.3", fc="white", ec="gray", alpha=0.8))
    
    # Plot positions
    axes[i, 1].plot(positions, color='green')
    axes[i, 1].set_ylabel("Position")
    axes[i, 1].set_title(f"Positions for Parameter Set {i+1}")
    axes[i, 1].grid(True)

plt.tight_layout(rect=[0, 0, 1, 0.96])  # Adjust for suptitle
plt.show()

# Run a verbose backtest with best parameters for basket 2
print("Running verbose backtest with best parameters for PICNIC_BASKET2:")
backtest_basket2(best_params_basket2["thresh"], best_params_basket2["position"], 
                best_params_basket2["std_window"], best_params_basket2["ma_window"], verbose=True)

# Compare the results of all three strategies
print("\n" + "="*80)
print("COMPARISON OF ALL STRATEGIES")
print("="*80)

print("1. PICNIC_BASKET1:")
print("  A. JAM & CROISSANT Direction Strategy - Best PnL: {:.2f}".format(best_pnl))
print(f"     Parameters: Threshold={best_params['thresh']}, StdWindow={best_params['std_window']}, MAWindow={best_params['ma_window']}")

print("\n2. PICNIC_BASKET2:")
print("  A. JAM & CROISSANT Direction Strategy - Best PnL: {:.2f}".format(best_pnl_basket2))
print(f"     Parameters: Threshold={best_params_basket2['thresh']}, StdWindow={best_params_basket2['std_window']}, MAWindow={best_params_basket2['ma_window']}")

# Create a final bar chart comparing the strategies
strategies = [
    "JAM & CROISSANT Direction \n PICNIC_BASKET1",
    "JAM & CROISSANT Direction \n PICNIC_BASKET2"
]

pnls = [best_pnl, best_pnl_basket2]

plt.figure(figsize=(12, 8))
bars = plt.bar(strategies, pnls, color=['blue', 'green'])

# Add PnL values on top of bars
for bar in bars:
    height = bar.get_height()
    plt.text(bar.get_x() + bar.get_width()/2., height + 0.1,
            f'{height:.2f}',
            ha='center', va='bottom', fontweight='bold')

plt.title('Comparison of Best PnL Across Strategies', fontsize=16)
plt.ylabel('PnL', fontsize=14)
plt.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

# Create a comprehensive final recommendations table
print("\n" + "="*80)
print("FINAL RECOMMENDATIONS")
print("="*80)

print("Based on the backtesting results, here are the recommendations:")

# For PICNIC_BASKET1
jc_basket1_result = sorted_results[0]
print("\n1. For PICNIC_BASKET1:")
print(f"   Strategy: Trade when JAM & CROISSANT move in the same direction")
print(f"   Threshold: {jc_basket1_result['thresh']}")
print(f"   Standard Deviation Window: {jc_basket1_result['std_window']}")
print(f"   Moving Average Window: {jc_basket1_result['ma_window']}")
print(f"   Expected PnL: {jc_basket1_result['final_pnl']:.2f}")
print(f"   Max Drawdown: {jc_basket1_result['max_drawdown']:.2f}")
print(f"   Number of Orders: {jc_basket1_result['order_count']}")

# For PICNIC_BASKET2
jc_basket2_result = sorted_results_basket2[0]
print("\n2. For PICNIC_BASKET2:")
print(f"   Strategy: Trade when JAM & CROISSANT move in the same direction")
print(f"   Threshold: {jc_basket2_result['thresh']}")
print(f"   Standard Deviation Window: {jc_basket2_result['std_window']}")
print(f"   Moving Average Window: {jc_basket2_result['ma_window']}")
print(f"   Expected PnL: {jc_basket2_result['final_pnl']:.2f}")
print(f"   Max Drawdown: {jc_basket2_result['max_drawdown']:.2f}")
print(f"   Number of Orders: {jc_basket2_result['order_count']}")

print("\nImplementation Notes:")
print("1. The strategy shows that filtering trades based on JAM & CROISSANT moving in the same direction")
print("   can improve trading performance by reducing false signals.")
print("2. The Maximum Drawdown figures should be considered when allocating capital to this strategy.")
print("3. Consider implementing a stop-loss mechanism to protect against unexpected moves.")
print("4. The strategy's performance could potentially be improved by:")
print("   - Adjusting position sizes based on the strength of the signal (z-score magnitude)")
print("   - Adding time-based filters to avoid trading during certain periods")
print("   - Implementing dynamic thresholds that adapt to market volatility")

In [ ]:
import pandas as pd
import numpy as np
from collections import deque
from dataclasses import dataclass
from typing import Dict, List
import plotly.graph_objects as go
from plotly.subplots import make_subplots

"""
PCA‑Factor back‑tester that works directly with the original IMC/Prosperity CSV **long** format
(one row per <timestamp, product>) and adds a simple latency + level‑1‑volume fill model.

The code assumes you already have the three day‑files in the working directory **and** that each
product appears the same number of times in the same chronological order (the case for the
competition data).  If the lengths differ the extra rows are truncated so all legs stay aligned.
"""

# --------------------------------------------------------------------------------------
# Helper dataclass for order‑book snapshot (top‑of‑book & smart‑weighted mid only)
# --------------------------------------------------------------------------------------
@dataclass
class Snap:
    bid: float
    ask: float
    bid_vol: int
    ask_vol: int
    mid: float
    swmid: float

# --------------------------------------------------------------------------------------
# Products & basket recipe
# --------------------------------------------------------------------------------------
CROISSANT, JAM, DJEMBE, BASKET = "CROISSANTS", "JAMS", "DJEMBES", "PICNIC_BASKET1"
WEIGHTS = {CROISSANT: 6, JAM: 3, DJEMBE: 1}
LEGS = [CROISSANT, JAM, DJEMBE]

# --------------------------------------------------------------------------------------
# Build a **wide** DataFrame (one row = one tick across all four products)
# --------------------------------------------------------------------------------------

def build_wide_df(csv_paths: List[str]) -> pd.DataFrame:
    df_long = pd.concat([pd.read_csv(p, sep=";", header=0) for p in csv_paths], ignore_index=True)

    # Keep only columns we need
    cols = [
        "bid_price_1", "bid_volume_1", "ask_price_1", "ask_volume_1",
    ]

    # Split by product and align by index (drop extras to equal length)
    prod_dfs = {}
    min_len = np.inf
    for prod in [*LEGS, BASKET]:
        prod_df = (
            df_long[df_long["product"] == prod]
            .sort_values(["day", "timestamp"], ignore_index=True)
            .reset_index(drop=True)
        )
        prod_dfs[prod] = prod_df
        min_len = min(min_len, len(prod_df))

    # Truncate to common length
    for prod in prod_dfs:
        prod_dfs[prod] = prod_dfs[prod].iloc[: int(min_len)].reset_index(drop=True)

    # Build wide
    wide = pd.DataFrame(index=range(int(min_len)))
    for prod, pdf in prod_dfs.items():
        prefix = prod.lower() + "_"
        for c in cols:
            wide[f"{prefix}{c}"] = pdf[c]
        # add mid & swmid
        bid = pdf["bid_price_1"].to_numpy()
        ask = pdf["ask_price_1"].to_numpy()
        bid_vol = np.abs(pdf["bid_volume_1"].to_numpy())
        ask_vol = np.abs(pdf["ask_volume_1"].to_numpy())
        mid = 0.5 * (bid + ask)
        swmid = (bid * ask_vol + ask * bid_vol) / (bid_vol + ask_vol)
        wide[f"{prefix}mid"] = mid
        wide[f"{prefix}swmid"] = swmid

    wide["continuous_time"] = np.arange(len(wide))
    return wide

# --------------------------------------------------------------------------------------
# Canonical snapshot from a wide‑format row (same logic as live trader)
# --------------------------------------------------------------------------------------

def row_to_snapshot(row: pd.Series) -> Dict[str, Snap]:
    snap: Dict[str, Snap] = {}
    for p in [*LEGS, BASKET]:
        prefix = f"{p.lower()}_"
        bid = row[f"{prefix}bid_price_1"]
        ask = row[f"{prefix}ask_price_1"]
        bid_vol = abs(row[f"{prefix}bid_volume_1"])
        ask_vol = abs(row[f"{prefix}ask_volume_1"])
        mid = row[f"{prefix}mid"]
        swmid = row[f"{prefix}swmid"]
        snap[p] = Snap(bid, ask, bid_vol, ask_vol, mid, swmid)
    return snap

# --------------------------------------------------------------------------------------
# PCA‑factor signal (identical maths to live PcaFactorTrader)
# --------------------------------------------------------------------------------------
class PCAFactorSignal:
    def __init__(self, window: int = 300, z_thresh: float = 2.0,
                 pos_lim: int = 60, refit_every: int = 50):
        self.window = window
        self.z_thresh = z_thresh
        self.pos_lim = pos_lim
        self.refit_every = refit_every
        self.buf: deque[np.ndarray] = deque(maxlen=window)
        # start with fundamental recipe weights normalised
        self.weights = np.array([6, 3, 1]) / 10
        self.ticks_since_refit = 0
        self.spread_hist: deque[float] = deque(maxlen=window)

    def update(self, swmids: np.ndarray, basket_swmid: float) -> int:
        """Return desired basket position (long +, short –)."""
        self.buf.append(swmids)
        if len(self.buf) < self.window:
            return 0  # warm‑up
        # periodic PCA refit
        self.ticks_since_refit += 1
        if self.ticks_since_refit >= self.refit_every:
            X = np.array(self.buf)
            X = X - X.mean(axis=0)
            _, _, vh = np.linalg.svd(X, full_matrices=False)
            self.weights = vh[-1]
            self.ticks_since_refit = 0
        synth_price = float(self.weights @ swmids)
        spread = basket_swmid - synth_price
        self.spread_hist.append(spread)
        if len(self.spread_hist) < self.window:
            return 0
        mu = float(np.mean(self.spread_hist))
        sigma = float(np.std(self.spread_hist))
        if sigma == 0:
            return 0
        z = (spread - mu) / sigma
        if z > self.z_thresh:
            return -self.pos_lim
        elif z < -self.z_thresh:
            return self.pos_lim
        return 0

# --------------------------------------------------------------------------------------
# Latency & volume‑aware fill model
# --------------------------------------------------------------------------------------
@dataclass
class PendingOrder:
    product: str
    side: int   # +1 buy, -1 sell
    qty: int
    submit_idx: int
    fill_idx: int


def simulate_fill(order: "PendingOrder", snap: Dict[str, Snap]) -> int:
    """Return actual filled quantity (can be partial)."""
    ob = snap[order.product]
    vol_avail = ob.ask_vol if order.side > 0 else ob.bid_vol
    return min(order.qty, vol_avail)

# --------------------------------------------------------------------------------------
# Back‑test harness with latency + top‑of‑book volume cap
# --------------------------------------------------------------------------------------

def backtest(csv_paths: List[str], signal: PCAFactorSignal,
             latency_ticks: int = 1):
    """Run the strategy with a simple latency & volume‑cap fill model."""
    df = build_wide_df(csv_paths)

    position = 0  # basket inventory only (legs hedge immediately)
    cash = 0.0
    pnl_hist, spread_hist, synth_hist = [], [], []
    pending: List[PendingOrder] = []

    for idx, row in df.iterrows():
        snap = row_to_snapshot(row)

        # ---- fill pending orders whose latency has expired ----
        filled_orders = [po for po in pending if po.fill_idx == idx]
        for po in filled_orders:
            filled = simulate_fill(po, snap)
            if filled == 0:
                continue
            price = snap[po.product].ask if po.side > 0 else snap[po.product].bid
            cash -= po.side * filled * price  # buy (side=+1) decreases cash
            if po.product == BASKET:
                position += po.side * filled
            po.qty -= filled
        # keep only live orders
        pending = [po for po in pending if po.qty > 0 and po.fill_idx >= idx]

        # ---- new signal ----
        swmids = np.array([snap[p].swmid for p in LEGS])
        basket_swmid = snap[BASKET].swmid
        target = signal.update(swmids, basket_swmid)
        delta = target - position
        if delta != 0:
            side = 1 if delta > 0 else -1
            qty = abs(delta)
            # enqueue basket & hedge orders
            pending.append(PendingOrder(BASKET, side, qty, idx, idx + latency_ticks))
            for leg, w in WEIGHTS.items():
                pending.append(PendingOrder(leg, -side, qty * w, idx, idx + latency_ticks))

        # ---- mark‑to‑market ----
        pnl_hist.append(cash + position * basket_swmid)
        synth_price = float(signal.weights @ swmids)
        spread_hist.append(basket_swmid - synth_price)
        synth_hist.append(synth_price)

    return df, np.array(pnl_hist), np.array(spread_hist), np.array(synth_hist)

# --------------------------------------------------------------------------------------
# Example usage (re‑run this cell in your notebook)
# --------------------------------------------------------------------------------------
if __name__ == "__main__":
    paths = [
        "prices_round_2_day_-1.csv",
        "prices_round_2_day_0.csv",
        "prices_round_2_day_1.csv",
    ]

    sig = PCAFactorSignal(window=300, z_thresh=2.0, pos_lim=60, refit_every=50)
    df_wide, pnl, spread, synth = backtest(paths, sig, latency_ticks=1)

    # ---------- Plot ----------
    fig = make_subplots(
        rows=2, cols=1,
        specs=[[{"secondary_y": True}], [{"secondary_y": False}]],
        shared_xaxes=True,
        vertical_spacing=0.08,
    )

    fig.add_trace(
        go.Scatter(x=df_wide["continuous_time"], y=df_wide[f"{BASKET.lower()}_swmid"],
                    mode="lines", name="Basket SWMID", line=dict(width=2, color="blue")),
        row=1, col=1, secondary_y=False,
    )
    fig.add_trace(
        go.Scatter(x=df_wide["continuous_time"], y=synth,
                    mode="lines", name="Synthetic PCA SWMID", line=dict(width=2, color="red")),
        row=1, col=1, secondary_y=True,
    )

    fig.add_trace(
        go.Scatter(x=df_wide["continuous_time"], y=spread,
                    mode="lines", name="Spread", line=dict(width=1, color="orange")),
        row=2, col=1,
    )
    fig.add_trace(
        go.Scatter(x=df_wide["continuous_time"], y=pnl,
                    mode="lines", name="PnL", line=dict(width=2, color="green")),
        row=2, col=1,
    )

    fig.update_layout(title="PCA Factor Trader – Latency & Volume‑Aware Back‑test",
                      template="plotly_white", height=800)
    fig.update_xaxes(title_text="Tick", row=2, col=1)
    fig.update_yaxes(title_text="Basket SWMID", row=1, col=1, secondary_y=False)
    fig.update_yaxes(title_text="Synthetic SWMID", row=1, col=1, secondary_y=True)
    fig.update_yaxes(title_text="Spread / PnL", row=2, col=1)
    fig.show()

    # ---------- Summary ----------
    print("Final PnL: {:.2f}".format(pnl[-1]))
    max_dd = np.max(np.maximum.accumulate(pnl) - pnl)
    sharpe = (np.mean(np.diff(pnl)) / (np.std(np.diff(pnl)) + 1e-9)) * np.sqrt(252 * 6.5 * 60)
    print(f"Max Drawdown: {max_dd:.2f}")
    print(f"Sharpe (tick‑level): {sharpe:.2f}")


In [ ]:
import pandas as pd
import numpy as np
from collections import deque
from dataclasses import dataclass
from typing import Dict, List, Tuple
import itertools
import plotly.graph_objects as go
from plotly.subplots import make_subplots

"""
Fixed‑Weight Spread Mean‑Reversion Back‑tester
================================================
This **back‑to‑basics** version removes PCA and uses the contract‑spec
weights (6 × CROISSANTS + 3 × JAMS + 1 × DJEMBE) to build a synthetic
basket.  A simple z‑score trigger with hysteresis drives the position.
The file still includes:
  • Latency & level‑1‑volume fill model.
  • Grid‑search over (window, z_entry, z_exit, pos_lim, latency).
  • Plotly visualisation of Basket vs Synthetic, Spread, and P&L.
"""

# --------------------------------------------------------------------------------------
# Helper dataclass for order‑book snapshot (top‑of‑book & smart‑weighted mid only)
# --------------------------------------------------------------------------------------
@dataclass
class Snap:
    bid: float
    ask: float
    bid_vol: int
    ask_vol: int
    mid: float
    swmid: float

# --------------------------------------------------------------------------------------
# Products & basket recipe
# --------------------------------------------------------------------------------------
CROISSANT, JAM, DJEMBE, BASKET = "CROISSANTS", "JAMS", "DJEMBES", "PICNIC_BASKET1"
WEIGHTS = {CROISSANT: 6, JAM: 3, DJEMBE: 1}
LEGS = [CROISSANT, JAM, DJEMBE]

# --------------------------------------------------------------------------------------
# Build a **wide** DataFrame (one row = one tick across all four products)
# --------------------------------------------------------------------------------------

def build_wide_df(csv_paths: List[str]) -> pd.DataFrame:
    df_long = pd.concat([pd.read_csv(p, sep=";", header=0) for p in csv_paths], ignore_index=True)

    cols = ["bid_price_1", "bid_volume_1", "ask_price_1", "ask_volume_1"]

    prod_dfs = {}
    min_len = np.inf
    for prod in [*LEGS, BASKET]:
        pdf = (
            df_long[df_long["product"] == prod]
            .sort_values(["day", "timestamp"], ignore_index=True)
            .reset_index(drop=True)
        )
        prod_dfs[prod] = pdf
        min_len = min(min_len, len(pdf))

    for prod in prod_dfs:
        prod_dfs[prod] = prod_dfs[prod].iloc[: int(min_len)].reset_index(drop=True)

    wide = pd.DataFrame(index=range(int(min_len)))
    for prod, pdf in prod_dfs.items():
        prefix = prod.lower() + "_"
        for c in cols:
            wide[f"{prefix}{c}"] = pdf[c]
        bid = pdf["bid_price_1"].to_numpy()
        ask = pdf["ask_price_1"].to_numpy()
        bid_vol = np.abs(pdf["bid_volume_1"].to_numpy())
        ask_vol = np.abs(pdf["ask_volume_1"].to_numpy())
        mid = 0.5 * (bid + ask)
        swmid = (bid * ask_vol + ask * bid_vol) / (bid_vol + ask_vol)
        wide[f"{prefix}mid"] = mid
        wide[f"{prefix}swmid"] = swmid

    wide["continuous_time"] = np.arange(len(wide))
    return wide

# --------------------------------------------------------------------------------------
# Canonical snapshot from a wide‑format row
# --------------------------------------------------------------------------------------

def row_to_snapshot(row: pd.Series) -> Dict[str, Snap]:
    snap: Dict[str, Snap] = {}
    for p in [*LEGS, BASKET]:
        prefix = f"{p.lower()}_"
        bid = row[f"{prefix}bid_price_1"]
        ask = row[f"{prefix}ask_price_1"]
        bid_vol = abs(row[f"{prefix}bid_volume_1"])
        ask_vol = abs(row[f"{prefix}ask_volume_1"])
        mid = row[f"{prefix}mid"]
        swmid = row[f"{prefix}swmid"]
        snap[p] = Snap(bid, ask, bid_vol, ask_vol, mid, swmid)
    return snap

# --------------------------------------------------------------------------------------
# Fixed‑weight spread mean‑reversion signal with hysteresis
# --------------------------------------------------------------------------------------
class FixedSpreadSignal:
    def __init__(self, window: int = 300, z_entry: float = 2.0,
                 z_exit: float = 0.6, pos_lim: int = 60):
        self.window = window
        self.z_entry = z_entry
        self.z_exit = z_exit
        self.pos_lim = pos_lim
        self.spread_hist: deque[float] = deque(maxlen=window)

    def update(self, basket_swmid: float, synthetic_swmid: float,
               current_pos: int) -> int:
        """Return desired basket position given current inventory."""
        spread = basket_swmid - synthetic_swmid
        self.spread_hist.append(spread)
        if len(self.spread_hist) < self.window:
            return 0  # warm‑up
        mu = float(np.mean(self.spread_hist))
        sigma = float(np.std(self.spread_hist))
        if sigma == 0:
            return 0
        z = (spread - mu) / sigma
        if current_pos == 0:
            if z > self.z_entry:
                return -self.pos_lim
            elif z < -self.z_entry:
                return self.pos_lim
            else:
                return 0
        else:
            # already in a trade – exit when z returns inside the band
            if abs(z) < self.z_exit:
                return 0
            else:
                return current_pos

# --------------------------------------------------------------------------------------
# Latency & volume‑aware fill model
# --------------------------------------------------------------------------------------
@dataclass
class PendingOrder:
    product: str
    side: int   # +1 buy, -1 sell
    qty: int
    submit_idx: int
    fill_idx: int


def simulate_fill(order: "PendingOrder", snap: Dict[str, Snap]) -> int:
    ob = snap[order.product]
    vol_avail = ob.ask_vol if order.side > 0 else ob.bid_vol
    return min(order.qty, vol_avail)

# --------------------------------------------------------------------------------------
# Back‑test harness
# --------------------------------------------------------------------------------------

def backtest(df: pd.DataFrame, signal: FixedSpreadSignal,
             latency_ticks: int = 1) -> Tuple[np.ndarray, np.ndarray, np.ndarray]:
    position = 0  # basket inventory only
    cash = 0.0
    pnl_hist, spread_hist, synth_hist = [], [], []
    pending: List[PendingOrder] = []

    for idx, row in df.iterrows():
        snap = row_to_snapshot(row)

        # ---- fill pending orders ----
        filled_orders = [po for po in pending if po.fill_idx == idx]
        for po in filled_orders:
            filled = simulate_fill(po, snap)
            if filled == 0:
                continue
            price = snap[po.product].ask if po.side > 0 else snap[po.product].bid
            cash -= po.side * filled * price
            if po.product == BASKET:
                position += po.side * filled
            po.qty -= filled
        pending = [po for po in pending if po.qty > 0 and po.fill_idx >= idx]

        # ---- compute synthetic price ----
        synth_price = sum(WEIGHTS[p] * snap[p].swmid for p in LEGS)

        # ---- new target position ----
        target = signal.update(snap[BASKET].swmid, synth_price, position)
        delta = target - position
        if delta != 0:
            side = 1 if delta > 0 else -1
            qty = abs(delta)
            # enqueue basket & hedge orders
            pending.append(PendingOrder(BASKET, side, qty, idx, idx + latency_ticks))
            for leg, w in WEIGHTS.items():
                pending.append(PendingOrder(leg, -side, qty * w, idx, idx + latency_ticks))

        # ---- mark‑to‑market ----
        pnl_hist.append(cash + position * snap[BASKET].swmid)
        spread_hist.append(snap[BASKET].swmid - synth_price)
        synth_hist.append(synth_price)

    return np.array(pnl_hist), np.array(spread_hist), np.array(synth_hist)

# --------------------------------------------------------------------------------------
# Performance metrics
# --------------------------------------------------------------------------------------

def max_drawdown(series: np.ndarray) -> float:
    running_max = np.maximum.accumulate(series)
    return float(np.max(running_max - series))


def sharpe_ratio(pnl: np.ndarray) -> float:
    ret = np.diff(pnl)
    if np.std(ret) == 0:
        return 0.0
    return float(np.mean(ret) / np.std(ret)) * np.sqrt(252 * 6.5 * 60)

# --------------------------------------------------------------------------------------
# Grid‑search helper
# --------------------------------------------------------------------------------------

def grid_search(df: pd.DataFrame,
                windows: List[int],
                z_entries: List[float],
                z_exits: List[float],
                pos_lims: List[int],
                latency_ticks: List[int],
                objective: str = "pnl") -> pd.DataFrame:
    results = []
    combos = list(itertools.product(windows, z_entries, z_exits, pos_lims, latency_ticks))
    print(f"Running grid search over {len(combos)} combinations …")

    for win, ze, zx, pos, lat in combos:
        sig = FixedSpreadSignal(window=win, z_entry=ze, z_exit=zx, pos_lim=pos)
        pnl, spread, synth = backtest(df, sig, latency_ticks=lat)
        results.append({
            "window": win,
            "z_entry": ze,
            "z_exit": zx,
            "pos_lim": pos,
            "latency": lat,
            "final_pnl": pnl[-1],
            "max_dd": max_drawdown(pnl),
            "sharpe": sharpe_ratio(pnl),
        })

    res_df = pd.DataFrame(results)
    if objective == "pnl":
        res_df = res_df.sort_values("final_pnl", ascending=False)
    elif objective == "sharpe":
        res_df = res_df.sort_values("sharpe", ascending=False)
    elif objective == "drawdown":
        res_df = res_df.sort_values("max_dd")
    return res_df.reset_index(drop=True)

# --------------------------------------------------------------------------------------
# Example usage (run in notebook or as script)
# --------------------------------------------------------------------------------------
if __name__ == "__main__":
    paths = [
        "prices_round_2_day_-1.csv",
        "prices_round_2_day_0.csv",
        "prices_round_2_day_1.csv",
    ]

    df_wide = build_wide_df(paths)

    # ------------- Parameter grid -------------
    windows = [150, 300, 450]
    z_entries = [1.5, 2.0, 2.5]
    z_exits = [0.5, 0.7, 1.0]
    pos_lims = [40, 60, 80]
    latency_list = [0, 1, 2]

    res = grid_search(df_wide, windows, z_entries, z_exits, pos_lims, latency_list, objective="pnl")

    print("\nTop 10 parameter sets by final PnL:")
    print(res.head(10).to_string(index=False))

    best = res.iloc[0]
    sig_best = FixedSpreadSignal(window=int(best.window), z_entry=float(best.z_entry),
                                 z_exit=float(best.z_exit), pos_lim=int(best.pos_lim))
    pnl_best, spread_best, synth_best = backtest(df_wide, sig_best, latency_ticks=int(best.latency))

    fig = make_subplots(
        rows=2, cols=1,
        specs=[[{"secondary_y": True}], [{"secondary_y": False}]],
        shared_xaxes=True,
        vertical_spacing=0.08,
    )

    fig.add_trace(
        go.Scatter(x=df_wide["continuous_time"], y=df_wide[f"{BASKET.lower()}_swmid"],
                    mode="lines", name="Basket SWMID", line=dict(width=2, color="blue")),
        row=1, col=1, secondary_y=False,
    )
    fig.add_trace(
        go.Scatter(x=df_wide["continuous_time"], y=synth_best,
                    mode="lines", name="Synthetic SWMID", line=dict(width=2, color="red")),
        row=1, col=1, secondary_y=True,
    )

    fig.add_trace(
        go.Scatter(x=df_wide["continuous_time"], y=spread_best,
                    mode="lines", name="Spread", line=dict(width=1, color="orange")),
        row=2, col=1,
    )
    fig.add_trace(
        go.Scatter(x=df_wide["continuous_time"], y=pnl_best,
                    mode="lines", name="PnL", line=dict(width=2, color="green")),
        row=2, col=1,
    )

    fig.update_layout(title="Best Fixed‑Weight Spread Trader – Grid‑Search Result",
                      template="plotly_white", height=800)
    fig.update_xaxes(title_text="Tick", row=2, col=1)
    fig.update_yaxes(title_text="Basket SWMID", row=1, col=1, secondary_y=False)
    fig.update_yaxes(title_text="Synthetic SWMID", row=1, col=1, secondary_y=True)
    fig.update_yaxes(title_text="Spread / PnL", row=2, col=1)
    fig.show()

    print("\nBest parameter set:")
    print(best.to_string())
    print("Final PnL: {:.2f}".format(pnl_best[-1]))
    print("Max Drawdown: {:.2f}".format(max_drawdown(pnl_best)))
    print("Sharpe (tick‑level): {:.2f}".format(sharpe_ratio(pnl_best)))
